In [0]:
# Silver: explode nested Prometheus JSON to one row per sample, keep useful labels,
# build a stable series key, drop bad values, dedupe overlapping runs (latest ingest wins).
from pyspark.sql import functions as F, Window

KEEP = ["name", "image", "instance", "job", "cpu", "device",
        "mountpoint", "fstype", "zone", "type"]

bronze = spark.table("homelab.bronze.prom_raw")

# one row per series, then one row per (timestamp, value) pair
points = (
    bronze
    .select("metric_name", "_ingest_ts", F.explode("response.data.result").alias("s"))
    .select(
        "metric_name", "_ingest_ts",
        F.map_filter("s.metric", lambda k, v: k.isin(KEEP)).alias("labels"),
        F.explode("s.values").alias("p"),
    )
)

# typed columns + stable series key (hash of sorted label=value pairs)
typed = points.select(
    "metric_name",
    F.sha2(F.concat_ws("|", F.array_sort(F.transform(
        F.map_entries("labels"), lambda e: F.concat_ws("=", e["key"], e["value"])
    ))), 256).alias("series_key"),
    "labels",
    F.timestamp_seconds(F.col("p")[0].cast("double")).alias("ts"),
    F.col("p")[1].cast("double").alias("value"),
    "_ingest_ts",
)

# drop NaN / +-Inf / unparseable values
clean = typed.filter(
    F.col("value").isNotNull() & ~F.isnan("value") & F.col("value").between(-1e300, 1e300)
)

# dedupe: same metric + series + timestamp -> keep the most recently ingested copy
w = Window.partitionBy("metric_name", "series_key", "ts").orderBy(F.col("_ingest_ts").desc())
silver = (
    clean.withColumn("rn", F.row_number().over(w))
    .filter("rn = 1").drop("rn", "_ingest_ts")
    .withColumn("dt", F.to_date("ts"))
)

(silver.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("homelab.silver.prom_samples"))

In [0]:
dups = (spark.table("homelab.silver.prom_samples")
        .groupBy("metric_name", "series_key", "ts").count()
        .filter("count > 1").limit(1).count())
assert dups == 0, "silver has duplicate (metric, series, ts) rows"